# Benchmark Completo — Transfer Learning no CIFAR-10 (versão atual)
**Dataset COMPLETO 50.000 imagens · 10 épocas · GPU**

Os mesmos 4 experimentos da versão de 5k, agora com o dataset completo: mostra quanto cada estratégia ganha com 10× mais dados. Mesma configuração dos labs: backbone **100% congelado** (inclusive BatchNorm) na Feature Extraction e no Fine-Tuning Parcial, `lr=1e-2` na `fc`, sem Data Augmentation.

| # | Estratégia | Params treináveis |
|---|---|---|
| 1 | Scratch (do zero) | 11.2M |
| 2 | Feature Extraction | 5.1K |
| 3 | Fine-Tuning Parcial (layer4) | 8.4M |
| 4 | Fine-Tuning Completo | 11.2M |

> Os resultados e tempos salvos neste notebook foram gerados numa GPU local (NVIDIA RTX 5050), não numa T4 do Colab: a acurácia é comparável, os tempos não.

## ▶️ Passo 0 — Ativar a GPU

**Ambiente de execução → Alterar o tipo de ambiente de execução → GPU (T4)**

Execute a célula abaixo para confirmar. Sem GPU este notebook levaria mais de 10 horas.

In [1]:
import torch

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'✅ GPU ativa: {torch.cuda.get_device_name(0)}')
else:
    device = torch.device('cpu')
    print('⚠️  GPU NÃO detectada — ative antes de continuar!')
    raise SystemExit('Interrompido: GPU obrigatória para este notebook.')

✅ GPU ativa: NVIDIA GeForce RTX 5050 Laptop GPU


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import time

torch.manual_seed(42)

NUM_EPOCHS = 10
N_TRAIN    = 50000   # dataset completo
N_VAL      = 10000   # validação completa
BATCH      = 128     # batch maior aproveita melhor a GPU com dataset grande

results = {}
print(f'Config: {N_TRAIN} treino | {N_VAL} val | {NUM_EPOCHS} épocas | batch={BATCH} | device={device}')

Config: 50000 treino | 10000 val | 10 épocas | batch=128 | device=cuda


In [3]:
transform_std = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Dataset completo — sem subset
# ── Download rápido do CIFAR-10 ────────────────────────────────────────────
# O servidor oficial (Univ. de Toronto) está muito lento. Baixamos o MESMO arquivo de
# um espelho rápido; o torchvision confere o MD5 e, se não bater (ou se o espelho
# estiver fora do ar), baixa do servidor oficial automaticamente.
import os, urllib.request
if not os.path.exists('data/cifar-10-batches-py'):
    os.makedirs('data', exist_ok=True)
    try:
        urllib.request.urlretrieve(
            'https://data.brainchip.com/dataset-mirror/cifar10/cifar-10-python.tar.gz',
            'data/cifar-10-python.tar.gz')
    except Exception as e:
        print(f'Espelho indisponível ({e}); usando o servidor oficial.')

train_full_std = datasets.CIFAR10(root='data', train=True,  download=True, transform=transform_std)
val_full       = datasets.CIFAR10(root='data', train=False, download=True, transform=transform_std)

train_loader_std = DataLoader(train_full_std, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
val_loader       = DataLoader(val_full,       batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

print(f'✅ Dataset completo: {len(train_full_std)} treino | {len(val_full)} val')

✅ Dataset completo: 50000 treino | 10000 val


In [4]:
criterion = nn.CrossEntropyLoss()

def train_epoch(model, loader, optimizer, freeze_bn=False):
    model.train()
    if freeze_bn:   # backbone pré-treinado 100% congelado (estatísticas da BatchNorm do ImageNet)
        for mod in model.modules():
            if isinstance(mod, nn.BatchNorm2d):
                mod.eval()
    running_loss = 0.0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(inputs), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
    return running_loss / len(loader.dataset)

def evaluate(model, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            _, preds = torch.max(model(inputs), 1)
            correct += (preds == labels).sum().item()
    return correct / len(loader.dataset)

def run_experiment(name, model, optimizer, train_loader, n_epochs=NUM_EPOCHS, freeze_bn=False):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'\n{"="*55}')
    print(f'  {name}')
    print(f'  Parâmetros treináveis: {trainable:,}')
    print(f'{"="*55}')
    t0 = time.time()
    acc = 0.0
    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(model, train_loader, optimizer, freeze_bn)
        acc  = evaluate(model, val_loader)
        elapsed_ep = time.time() - t0
        print(f'  Época {epoch:>2}/{n_epochs} | Loss: {loss:.4f} | Val Acc: {acc*100:.2f}% | {elapsed_ep/60:.1f}min acumulado')
    elapsed = time.time() - t0
    print(f'  ✅ {elapsed/60:.1f} min total | Acurácia final: {acc*100:.2f}%')
    results[name] = {'acc': acc * 100, 'time_min': elapsed / 60, 'params': trainable}
    return model

print('✅ Funções auxiliares definidas.')

✅ Funções auxiliares definidas.


## Experimento 1 — Scratch (do zero)
ResNet-18 com pesos **aleatórios**, sem pré-treinamento. Linha de base inferior: mostra o custo de não usar Transfer Learning.

In [5]:
torch.manual_seed(42)
m = models.resnet18(weights=None)
m.fc = nn.Linear(m.fc.in_features, 10)
m = m.to(device)
opt = optim.SGD(m.parameters(), lr=0.01, momentum=0.9)
_ = run_experiment('Scratch (do zero)', m, opt, train_loader_std)


  Scratch (do zero)
  Parâmetros treináveis: 11,181,642


  Época  1/10 | Loss: 1.4922 | Val Acc: 52.31% | 1.7min acumulado


  Época  2/10 | Loss: 0.9650 | Val Acc: 65.92% | 3.5min acumulado


  Época  3/10 | Loss: 0.7058 | Val Acc: 73.16% | 5.3min acumulado


  Época  4/10 | Loss: 0.5336 | Val Acc: 64.36% | 7.1min acumulado


...

  Época  6/10 | Loss: 0.2702 | Val Acc: 73.24% | 10.7min acumulado


  Época  7/10 | Loss: 0.1608 | Val Acc: 66.66% | 12.5min acumulado


  Época  8/10 | Loss: 0.1089 | Val Acc: 68.45% | 14.2min acumulado


  Época  9/10 | Loss: 0.0659 | Val Acc: 73.29% | 16.0min acumulado


  Época 10/10 | Loss: 0.0363 | Val Acc: 76.55% | 17.8min acumulado
  ✅ 17.8 min total | Acurácia final: 76.55%


## Experimento 2 — Feature Extraction
Backbone **100% congelado** (pesos e BatchNorm do ImageNet); treina apenas a camada `fc` (5.130 parâmetros). Estratégia do Lab 1.

In [6]:
torch.manual_seed(42)
m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
for p in m.parameters():
    p.requires_grad = False
m.fc = nn.Linear(m.fc.in_features, 10)
m = m.to(device)
opt = optim.SGD(m.fc.parameters(), lr=0.01, momentum=0.9)
_ = run_experiment('Feature Extraction', m, opt, train_loader_std, freeze_bn=True)


  Feature Extraction
  Parâmetros treináveis: 5,130


  Época  1/10 | Loss: 0.5205 | Val Acc: 85.28% | 0.7min acumulado


  Época  2/10 | Loss: 0.3839 | Val Acc: 86.47% | 1.5min acumulado


  Época  3/10 | Loss: 0.3637 | Val Acc: 86.33% | 2.3min acumulado


  Época  4/10 | Loss: 0.3524 | Val Acc: 85.27% | 3.0min acumulado


...

  Época  6/10 | Loss: 0.3400 | Val Acc: 87.04% | 4.5min acumulado


  Época  7/10 | Loss: 0.3319 | Val Acc: 86.74% | 5.3min acumulado


  Época  8/10 | Loss: 0.3292 | Val Acc: 86.70% | 6.0min acumulado


  Época  9/10 | Loss: 0.3252 | Val Acc: 86.85% | 6.8min acumulado


  Época 10/10 | Loss: 0.3249 | Val Acc: 86.89% | 7.6min acumulado
  ✅ 7.6 min total | Acurácia final: 86.89%


## Experimento 3 — Fine-Tuning Parcial (layer4)
Descongela apenas o último bloco residual (`layer4`, `lr=1e-4`) e treina a nova `fc` com `lr=1e-2`; o restante, inclusive a BatchNorm, fica congelado. Estratégia do Lab 2.

In [7]:
torch.manual_seed(42)
m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
for p in m.parameters():
    p.requires_grad = False
for p in m.layer4.parameters():
    p.requires_grad = True
m.fc = nn.Linear(m.fc.in_features, 10)
m = m.to(device)
opt = optim.SGD(
    [{'params': m.layer4.parameters(), 'lr': 1e-4},
     {'params': m.fc.parameters(),    'lr': 1e-2}],
    momentum=0.9)
_ = run_experiment('Fine-Tuning Parcial (layer4)', m, opt, train_loader_std, freeze_bn=True)


  Fine-Tuning Parcial (layer4)
  Parâmetros treináveis: 8,398,858


  Época  1/10 | Loss: 0.4699 | Val Acc: 86.86% | 0.9min acumulado


  Época  2/10 | Loss: 0.3109 | Val Acc: 89.05% | 1.8min acumulado


  Época  3/10 | Loss: 0.2749 | Val Acc: 89.43% | 3.1min acumulado


  Época  4/10 | Loss: 0.2523 | Val Acc: 89.75% | 4.4min acumulado


...

  Época  6/10 | Loss: 0.2171 | Val Acc: 90.35% | 7.2min acumulado


  Época  7/10 | Loss: 0.2024 | Val Acc: 90.20% | 8.6min acumulado


  Época  8/10 | Loss: 0.1906 | Val Acc: 90.58% | 9.4min acumulado


  Época  9/10 | Loss: 0.1795 | Val Acc: 90.88% | 10.3min acumulado


  Época 10/10 | Loss: 0.1667 | Val Acc: 90.71% | 11.1min acumulado
  ✅ 11.1 min total | Acurácia final: 90.71%


## Experimento 4 — Fine-Tuning Completo
Todas as camadas abertas para atualização (inclusive a BatchNorm), com `lr=0.001` para evitar *catastrophic forgetting*. Teto de desempenho com este dataset.

In [8]:
torch.manual_seed(42)
m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
m.fc = nn.Linear(m.fc.in_features, 10)
m = m.to(device)
opt = optim.SGD(m.parameters(), lr=0.001, momentum=0.9)
_ = run_experiment('Fine-Tuning Completo', m, opt, train_loader_std)


  Fine-Tuning Completo
  Parâmetros treináveis: 11,181,642


  Época  1/10 | Loss: 0.6718 | Val Acc: 90.58% | 1.7min acumulado


  Época  2/10 | Loss: 0.2338 | Val Acc: 92.60% | 3.4min acumulado


  Época  3/10 | Loss: 0.1564 | Val Acc: 93.35% | 5.2min acumulado


  Época  4/10 | Loss: 0.1117 | Val Acc: 93.89% | 6.9min acumulado


...

  Época  6/10 | Loss: 0.0569 | Val Acc: 94.08% | 10.3min acumulado


  Época  7/10 | Loss: 0.0403 | Val Acc: 94.32% | 12.0min acumulado


  Época  8/10 | Loss: 0.0284 | Val Acc: 94.33% | 13.7min acumulado


  Época  9/10 | Loss: 0.0217 | Val Acc: 94.32% | 15.5min acumulado


  Época 10/10 | Loss: 0.0159 | Val Acc: 94.44% | 17.2min acumulado
  ✅ 17.2 min total | Acurácia final: 94.44%


## Tabela de Resultados

In [9]:
print(f'\n{"="*65}')
print(f'{"BENCHMARK COMPLETO — CIFAR-10 · 50.000 imagens · 10 épocas":^65}')
print(f'{"="*65}')
print(f'{"Experimento":<30} {"Params":>12} {"Acurácia":>10} {"Tempo":>10}')
print(f'{"-"*65}')
for name, r in results.items():
    print(f'{name:<30} {r["params"]:>12,} {r["acc"]:>9.2f}% {r["time_min"]:>8.1f}min')
print(f'{"="*65}')
best = max(results, key=lambda k: results[k]['acc'])
print(f'  Melhor: {best} → {results[best]["acc"]:.2f}%')


   BENCHMARK COMPLETO — CIFAR-10 · 50.000 imagens · 10 épocas    
Experimento                          Params   Acurácia      Tempo
-----------------------------------------------------------------
Scratch (do zero)                11,181,642     76.55%     17.8min
Feature Extraction                    5,130     86.89%      7.6min
Fine-Tuning Parcial (layer4)      8,398,858     90.71%     11.1min
Fine-Tuning Completo             11,181,642     94.44%     17.2min
  Melhor: Fine-Tuning Completo → 94.44%
